In [5]:
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_squared_error
from xgboost import XGBRegressor
from sklearn.preprocessing import MinMaxScaler



In [6]:
class XGBoostRecommender:

    def __init__(self, csv_path):
        self.csv_path = csv_path
        self.model = None

    def load_data(self):
        return pd.read_csv(self.csv_path)

    def create_target(self, df):

        scaler = MinMaxScaler()

        df["rating_norm"] = scaler.fit_transform(
            df[["Google review rating"]]
        )

        df["duration_norm"] = scaler.fit_transform(
            df[["time needed to visit in hrs"]]
        )

        df["fee_norm"] = scaler.fit_transform(
            df[["Entrance Fee in INR"]]
        )

        df["recommendation_score"] = (
            0.5 * df["rating_norm"] +
            0.3 * df["duration_norm"] +
            0.2 * (1 - df["fee_norm"])
        )

        return df

    def preprocess(self, df):

        X = df[
            [
                "Google review rating",
                "time needed to visit in hrs",
                "Entrance Fee in INR",
                "Type",
                "City",
                "State"
            ]
        ]

        y = df["recommendation_score"]

        categorical = [
            "Type",
            "City",
            "State"
        ]

        numerical = [
            "Google review rating",
            "time needed to visit in hrs",
            "Entrance Fee in INR"
        ]

        preprocessor = ColumnTransformer(
            transformers=[
                (
                    "cat",
                    OneHotEncoder(handle_unknown="ignore"),
                    categorical
                ),
                (
                    "num",
                    "passthrough",
                    numerical
                )
            ]
        )

        return train_test_split(
            X,
            y,
            test_size=0.2,
            random_state=42
        ), preprocessor

    def train(self):

        df = self.load_data()

        df = self.create_target(df)

        (X_train, X_test, y_train, y_test), preprocessor = self.preprocess(df)

        self.model = Pipeline(
            steps=[
                ("preprocessor", preprocessor),
                (
                    "xgboost",
                    XGBRegressor(
                        n_estimators=200,
                        learning_rate=0.05,
                        max_depth=5,
                        random_state=42
                    )
                )
            ]
        )

        self.model.fit(X_train, y_train)

        predictions = self.model.predict(X_test)

        rmse = mean_squared_error(
            y_test,
            predictions
        ) ** 0.5

        print(f"RMSE : {rmse:.4f}")

    def save_model(self):
        joblib.dump(self.model, "xgboost_model.pkl")
        print("Model Saved")

    def load_model(self):
        self.model = joblib.load("xgboost_model.pkl")
        print("Model Loaded")

In [7]:

if __name__ == "__main__":

    recommender = XGBoostRecommender("data/cleaned_places.csv")

    recommender.train()

    recommender.save_model()

RMSE : 0.0200
Model Saved
